# Hack-A-Roo 2026, AI for Quantum track
# Notebook 3: Quantum kernel SVM

**Problem.** A support vector machine only needs a kernel, a function that says how similar two data points are. A quantum feature map turns a data point into a quantum state, and the overlap between two such states is a kernel. Does a quantum kernel classify real data as well as the classical RBF kernel, and what happens to it as the qubit count grows?

**What this notebook already does.** It loads a real tabular dataset, reduces it to one feature per qubit, builds a feature-map circuit gate by gate, computes the full quantum kernel matrix by exact statevector overlaps, trains an SVM on it, and compares against RBF and linear kernels on the same split. The data are split three ways: the training set fits the kernels and SVMs, a validation set chooses the one feature-map setting that is swept (the entangling scale), and a test set is touched exactly once at the end. It also reports two diagnostics of the kernel itself, uncentered kernel-target alignment and kernel concentration.

**Metric.** Test accuracy, quantum kernel versus RBF versus linear, on the same split, reported once. Secondary metric: uncentered kernel-target alignment on the training set.

**Hypothesis to test.** With a feature map tuned on validation data, the quantum kernel is competitive with RBF on this dataset. The test set here has 100 samples, so one percentage point is one sample. A difference of a few samples is not evidence of a systematic difference; repeating the split is.

**Deliverable for the poster.** One figure (the three kernel matrices side by side with their test accuracies) and one number per kernel. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes a few seconds. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Data. The dataset is reduced by PCA to QUBIT_COUNT features, one per qubit, then scaled into [0, FEATURE_SCALE].
QUBIT_COUNT = 4
TRAIN_SIZE = 150
VALIDATION_SIZE = 100                  # used only to choose ENTANGLING_SCALE from the candidates below
TEST_SIZE = 100                        # touched once, at the end
FEATURE_SCALE = 3.141592653589793      # pi. Features land in [0, pi]; the rz gates below use twice the feature, so those angles span [0, 2 pi].

# Feature map. See the circuit cell for exactly what these control.
FEATURE_MAP_REPETITIONS = 1
ENTANGLING_PATTERN = "linear"          # "linear" (i, i+1) or "full" (every pair)
ROTATION_SCALE = 1.0                   # multiplies every data-dependent angle
ENTANGLING_SCALE_CANDIDATES = [0.0, 0.1, 0.2, 0.5, 1.0]   # multiplies the pairwise angle only; chosen on validation accuracy. 0.0 is a product kernel. The textbook ZZ feature map uses 1.0 with 2 repetitions.

# SVM
SVM_REGULARIZATION = 1.0               # C
RBF_GAMMA = "scale"                    # "scale" means 1 / (feature count * feature variance), sklearn's default rule, computed explicitly below

# Output
OUTPUT_PATH = "quantum_kernel_dataset.pkl"
FIGURE_PATH = "quantum_kernel_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, scikit-learn 1.8 and 1.9.
!pip install -q "qiskit[visualization]" "scikit-learn>=1.5" matplotlib

In [ ]:
import time
import pickle
import numpy as np
import sklearn
import qiskit
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel

random_generator = np.random.default_rng(RANDOM_SEED)
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "scikit-learn": sklearn.__version__}
print("imports done", package_versions)

In [ ]:
# Data: the Wisconsin breast cancer dataset, 569 samples, 30 features, two classes.
# Standardize, PCA down to QUBIT_COUNT components, scale each component into [0, FEATURE_SCALE].
# PCA is fit on the training split only so the test split stays honest.

raw_features, labels = load_breast_cancer(return_X_y=True)
assert set(np.unique(labels)) == {0, 1}, "the alignment diagnostic below assumes binary 0/1 labels"
sample_indices = np.arange(len(labels))
train_indices, held_out_indices = train_test_split(
    sample_indices, train_size=TRAIN_SIZE, test_size=VALIDATION_SIZE + TEST_SIZE, random_state=RANDOM_SEED, stratify=labels)
validation_indices, test_indices = train_test_split(
    held_out_indices, train_size=VALIDATION_SIZE, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=labels[held_out_indices])
train_labels = labels[train_indices]
validation_labels = labels[validation_indices]
test_labels = labels[test_indices]

# Preprocessing is fit on the training rows only.
standardizer = StandardScaler().fit(raw_features[train_indices])
pca = PCA(n_components=QUBIT_COUNT, random_state=RANDOM_SEED).fit(standardizer.transform(raw_features[train_indices]))
range_scaler = MinMaxScaler(feature_range=(0, FEATURE_SCALE)).fit(pca.transform(standardizer.transform(raw_features[train_indices])))


def preprocess(raw_rows):
    """Standardize, project onto the PCA components, scale into [0, FEATURE_SCALE], using the training fit"""
    return range_scaler.transform(pca.transform(standardizer.transform(raw_rows)))


train_features = preprocess(raw_features[train_indices])
validation_features = preprocess(raw_features[validation_indices])
test_features = preprocess(raw_features[test_indices])
assert train_features.shape == (TRAIN_SIZE, QUBIT_COUNT)
assert validation_features.shape == (VALIDATION_SIZE, QUBIT_COUNT)
assert test_features.shape == (TEST_SIZE, QUBIT_COUNT)

print("variance explained by the", QUBIT_COUNT, "PCA components:", np.round(pca.explained_variance_ratio_, 3))
print("class balance, train / validation / test:", np.bincount(train_labels), np.bincount(validation_labels), np.bincount(test_labels))
print("validation and test features may fall slightly outside [0, pi] since the scaler was fit on train. That is fine.")

In [ ]:
# The feature map circuit, built gate by gate.
#
# For one repetition, on QUBIT_COUNT qubits with data vector x and entangling scale s:
#   1. H on every qubit.
#   2. rz(2 * ROTATION_SCALE * x_i) on qubit i.                        (single-feature phase)
#   3. rzz(2 * s * (pi - x_i) * (pi - x_j)) on pair (i, j).            (pairwise phase)
# Repeated FEATURE_MAP_REPETITIONS times. This is the structure of the common "ZZ feature map" but every
# angle and the entangling pattern are explicit here so you can change them.
#
# The kernel is K(x, y) = |<phi(x) | phi(y)>|^2, the probability that the state for x looks like the state for y.
# Because the circuit is a product of phases after the H layers, the (pi - x_i)(pi - x_j) term is what makes
# the kernel non-trivially entangled. With s = 0 the kernel is a product of 1-qubit kernels.

def entangling_pairs():
    """Lists the qubit pairs the feature map entangles, in order"""
    if ENTANGLING_PATTERN == "linear":
        return [(qubit_index, qubit_index + 1) for qubit_index in range(QUBIT_COUNT - 1)]
    if ENTANGLING_PATTERN == "full":
        return [(qubit_i, qubit_j) for qubit_i in range(QUBIT_COUNT) for qubit_j in range(qubit_i + 1, QUBIT_COUNT)]
    raise ValueError(f"unknown ENTANGLING_PATTERN {ENTANGLING_PATTERN}")


def build_feature_map_circuit(data_vector, entangling_scale):
    """Builds the feature map circuit for one data vector"""
    assert data_vector.shape == (QUBIT_COUNT,)
    circuit = QuantumCircuit(QUBIT_COUNT)
    for repetition_index in range(FEATURE_MAP_REPETITIONS):
        circuit.h(range(QUBIT_COUNT))
        for qubit_index in range(QUBIT_COUNT):
            circuit.rz(2 * ROTATION_SCALE * data_vector[qubit_index], qubit_index)
        for qubit_i, qubit_j in entangling_pairs():
            pairwise_angle = 2 * entangling_scale * (np.pi - data_vector[qubit_i]) * (np.pi - data_vector[qubit_j])
            circuit.rzz(pairwise_angle, qubit_i, qubit_j)
    return circuit


def feature_map_statevectors(feature_matrix, entangling_scale):
    """Returns an array of shape (samples, 2^n) holding the feature-map state of every row"""
    statevectors = np.zeros((len(feature_matrix), 2 ** QUBIT_COUNT), dtype=complex)
    for sample_index in range(len(feature_matrix)):
        statevectors[sample_index] = Statevector(build_feature_map_circuit(feature_matrix[sample_index], entangling_scale)).data
    return statevectors


def quantum_kernel_matrix(statevectors_a, statevectors_b):
    """Kernel matrix K[i, j] = |<a_i | b_j>|^2 from precomputed statevectors"""
    overlaps = statevectors_a.conj() @ statevectors_b.T
    return np.abs(overlaps) ** 2


print(build_feature_map_circuit(train_features[0], ENTANGLING_SCALE_CANDIDATES[-1]).draw(fold=120))
print("entangling pairs:", entangling_pairs())

In [ ]:
# Choose the entangling scale on the VALIDATION set, then compute the kernels the final SVM needs.
# The test set is not touched in this cell.

start_time = time.time()
validation_accuracy_by_scale = {}
off_diagonal_std_by_scale = {}
for candidate_scale in ENTANGLING_SCALE_CANDIDATES:
    candidate_train_statevectors = feature_map_statevectors(train_features, candidate_scale)
    candidate_validation_statevectors = feature_map_statevectors(validation_features, candidate_scale)
    candidate_train_kernel = quantum_kernel_matrix(candidate_train_statevectors, candidate_train_statevectors)
    candidate_validation_kernel = quantum_kernel_matrix(candidate_validation_statevectors, candidate_train_statevectors)
    candidate_svm = SVC(kernel="precomputed", C=SVM_REGULARIZATION).fit(candidate_train_kernel, train_labels)
    validation_accuracy_by_scale[candidate_scale] = candidate_svm.score(candidate_validation_kernel, validation_labels)
    off_diagonal_std_by_scale[candidate_scale] = candidate_train_kernel[~np.eye(TRAIN_SIZE, dtype=bool)].std()
print(f"validation sweep over {len(ENTANGLING_SCALE_CANDIDATES)} entangling scales took {time.time() - start_time:.1f}s")
print(f"{'entangling scale':>18s}{'validation acc':>16s}{'off-diag std':>14s}")
for candidate_scale in ENTANGLING_SCALE_CANDIDATES:
    print(f"{candidate_scale:18.2f}{validation_accuracy_by_scale[candidate_scale]:16.3f}{off_diagonal_std_by_scale[candidate_scale]:14.4f}")

# Ties go to the smallest scale, which is the simplest kernel.
ENTANGLING_SCALE = min(ENTANGLING_SCALE_CANDIDATES, key=lambda scale: (-validation_accuracy_by_scale[scale], scale))
print("chosen ENTANGLING_SCALE:", ENTANGLING_SCALE)

train_statevectors = feature_map_statevectors(train_features, ENTANGLING_SCALE)
test_statevectors = feature_map_statevectors(test_features, ENTANGLING_SCALE)
train_kernel = quantum_kernel_matrix(train_statevectors, train_statevectors)   # (train, train)
test_kernel = quantum_kernel_matrix(test_statevectors, train_statevectors)     # (test, train), what SVC.predict needs
assert train_kernel.shape == (TRAIN_SIZE, TRAIN_SIZE)
assert test_kernel.shape == (TEST_SIZE, TRAIN_SIZE)
assert np.allclose(np.diag(train_kernel), 1)         # every state overlaps itself perfectly
assert np.allclose(train_kernel, train_kernel.T)

# Diagnostic 1: uncentered kernel-target alignment. How much does the training kernel look like the ideal
# kernel y y^T where y is +1/-1? For a positive semidefinite kernel this is between 0 and 1. Higher means the kernel
# already separates the classes. It is a diagnostic, not a guarantee: a higher-alignment kernel need not give a
# more accurate SVM. For more than two classes this label kernel is wrong; see shred-out 3.
signed_labels = 2 * train_labels - 1
ideal_kernel = np.outer(signed_labels, signed_labels)


def kernel_target_alignment(kernel):
    """Normalized Frobenius inner product between a kernel and the ideal label kernel"""
    return np.sum(kernel * ideal_kernel) / (np.linalg.norm(kernel) * np.linalg.norm(ideal_kernel))


# Diagnostic 2: kernel concentration. If every off-diagonal entry is near the same value, the kernel
# cannot tell points apart and the SVM has nothing to work with. A small spread is a warning sign, not a proof
# that the SVM cannot generalize.
off_diagonal_mask = ~np.eye(TRAIN_SIZE, dtype=bool)
off_diagonal_entries = train_kernel[off_diagonal_mask]

print(f"uncentered kernel-target alignment of the chosen quantum kernel: {kernel_target_alignment(train_kernel):.4f}")
print(f"off-diagonal entries: mean {off_diagonal_entries.mean():.4f}, std {off_diagonal_entries.std():.4f}, "
      f"min {off_diagonal_entries.min():.4f}, max {off_diagonal_entries.max():.4f}")

In [ ]:
# Train three SVMs on the same training rows and score each ONCE on the test rows: quantum kernel (precomputed), RBF, linear.
# The RBF and linear kernels get no validation tuning here, so the comparison favors them slightly less than the
# quantum kernel; tuning C and gamma for them on the validation set is a fair thing to add.

quantum_svm = SVC(kernel="precomputed", C=SVM_REGULARIZATION).fit(train_kernel, train_labels)
# The RBF gamma is computed here rather than left to sklearn so the same number feeds both the SVM and the Gram matrix below.
if RBF_GAMMA == "scale":
    rbf_gamma = 1 / (QUBIT_COUNT * train_features.var())
else:
    rbf_gamma = RBF_GAMMA
rbf_svm = SVC(kernel="rbf", C=SVM_REGULARIZATION, gamma=rbf_gamma).fit(train_features, train_labels)
rbf_train_kernel = rbf_kernel(train_features, gamma=rbf_gamma)
linear_svm = SVC(kernel="linear", C=SVM_REGULARIZATION).fit(train_features, train_labels)

results = {
    "quantum": {
        "train_accuracy": quantum_svm.score(train_kernel, train_labels),
        "test_accuracy": quantum_svm.score(test_kernel, test_labels),
        "alignment": kernel_target_alignment(train_kernel),
    },
    "rbf": {
        "train_accuracy": rbf_svm.score(train_features, train_labels),
        "test_accuracy": rbf_svm.score(test_features, test_labels),
        "alignment": kernel_target_alignment(rbf_train_kernel),
    },
    "linear": {
        "train_accuracy": linear_svm.score(train_features, train_labels),
        "test_accuracy": linear_svm.score(test_features, test_labels),
        "alignment": kernel_target_alignment(train_features @ train_features.T),
    },
}

print("RESULTS, test set scored once")
print(f"{'kernel':10s}{'train acc':>12s}{'test acc':>12s}{'alignment':>12s}")
for kernel_name in results:
    print(f"{kernel_name:10s}{results[kernel_name]['train_accuracy']:12.3f}"
          f"{results[kernel_name]['test_accuracy']:12.3f}{results[kernel_name]['alignment']:12.4f}")
print(f"one test sample is {100 / TEST_SIZE:.1f} percentage points; differences of a few samples are within split-to-split variation")

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "QUBIT_COUNT": QUBIT_COUNT,
    "TRAIN_SIZE": TRAIN_SIZE,
    "VALIDATION_SIZE": VALIDATION_SIZE,
    "TEST_SIZE": TEST_SIZE,
    "FEATURE_SCALE": FEATURE_SCALE,
    "FEATURE_MAP_REPETITIONS": FEATURE_MAP_REPETITIONS,
    "ENTANGLING_PATTERN": ENTANGLING_PATTERN,
    "ROTATION_SCALE": ROTATION_SCALE,
    "ENTANGLING_SCALE_CANDIDATES": ENTANGLING_SCALE_CANDIDATES,
    "ENTANGLING_SCALE_CHOSEN": ENTANGLING_SCALE,
    "SVM_REGULARIZATION": SVM_REGULARIZATION,
    "RBF_GAMMA": RBF_GAMMA,
    "rbf_gamma_used": rbf_gamma,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "train_indices": train_indices,
    "validation_indices": validation_indices,
    "test_indices": test_indices,
    "standardizer": standardizer,
    "pca": pca,
    "range_scaler": range_scaler,
    "train_features": train_features,
    "validation_features": validation_features,
    "test_features": test_features,
    "train_labels": train_labels,
    "validation_labels": validation_labels,
    "test_labels": test_labels,
    "validation_accuracy_by_scale": validation_accuracy_by_scale,
    "train_kernel": train_kernel,
    "test_kernel": test_kernel,
    "results": results,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: the three training kernel matrices side by side, rows sorted by class so block structure is visible.

class_order = np.argsort(train_labels, kind="stable")
kernels_to_show = {
    "quantum": train_kernel[np.ix_(class_order, class_order)],
    "rbf": rbf_train_kernel[np.ix_(class_order, class_order)],
    "linear": (train_features @ train_features.T)[np.ix_(class_order, class_order)],
}

figure, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, kernel_name in zip(axes, kernels_to_show):
    image = axis.imshow(kernels_to_show[kernel_name], cmap="viridis")
    axis.set_title(f"{kernel_name} kernel, test acc {results[kernel_name]['test_accuracy']:.2f}")
    axis.set_xticks([])
    axis.set_yticks([])
    figure.colorbar(image, ax=axis, fraction=0.046)
figure.suptitle(f"Training kernels on {QUBIT_COUNT} qubits, entangling scale {ENTANGLING_SCALE} chosen on validation, rows and columns sorted by class")
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run is a small simulator experiment on one dataset. It establishes nothing about quantum advantage and does not predict which kernel wins elsewhere. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. Choose every setting on the validation split and report the test number once; repeat over several random splits and report the spread.

1. **Feature map design.** Sweep `ROTATION_SCALE`, `FEATURE_MAP_REPETITIONS`, and `ENTANGLING_PATTERN` alongside the entangling scale, all on validation accuracy. Start by reproducing the textbook setting (entangling scale 1.0, two repetitions) and look at the off-diagonal kernel entries; in this run that setting concentrated badly. Does alignment on the training set predict validation accuracy? Metric: test accuracy of the chosen setting, and the alignment-versus-validation-accuracy scatter over the grid. Entangling scale 0 is a product kernel with no entanglement, and it is a fair question whether entanglement helps at all here.

2. **Kernel concentration versus qubit count.** Raise `QUBIT_COUNT` from 2 to 10. Two things change at once: the PCA keeps more components (more input information) and the Hilbert space grows (more room to concentrate). Separate them by also running a version that keeps PCA at 4 components and pads the extra qubits with a constant angle. Does the off-diagonal std shrink, and does accuracy follow it or not? Metric: off-diagonal std and test accuracy versus qubit count, both variants. Concentration is not guaranteed to be monotonic for every dataset and map, so report what you see.

3. **Which datasets favor the quantum kernel.** Swap in `make_moons`, `make_circles`, `load_wine`, or `load_digits`. Two cautions. `make_moons` and `make_circles` have two raw features, so set `QUBIT_COUNT = 2` or add an explicit feature expansion before PCA, and pick `TRAIN_SIZE`, `VALIDATION_SIZE`, `TEST_SIZE` to fit the sample count. `load_wine` and `load_digits` have more than two classes, so the alignment diagnostic needs a multiclass label kernel (for example 1 when two labels agree and 0 otherwise, centered) or should be dropped; the SVM itself handles multiclass. Metric: test accuracy gap, quantum minus RBF, per dataset, averaged over five random splits with its spread.

4. **Shots instead of exact overlaps.** Replace the statevector overlap with a compute-uncompute circuit: apply the feature map for x, then the inverse feature map for y, and measure the probability of all zeros with `qiskit_aer` at a fixed shot count. The kernel matrix stops being exactly symmetric and positive semidefinite. How many shots does the SVM need to match the exact result, and what do you do about the broken kernel matrix? Metric: test accuracy versus shots.

5. **Projected quantum kernel.** Instead of the full state overlap, compute each qubit's single-qubit reduced density matrix from the feature-map state (`qiskit.quantum_info.partial_trace`), flatten those into a classical feature vector, and put an RBF kernel on top. Projected kernels can resist concentration in some settings and still concentrate in others. Metric: test accuracy and off-diagonal std versus qubit count, compared against the full overlap kernel from shred-out 2.

6. **Train the feature map.** Treat `ROTATION_SCALE` and the entangling scale as continuous parameters and optimize them to maximize alignment on the training set with `scipy.optimize.minimize`. Does maximizing alignment improve validation accuracy, or does it overfit? Metric: validation and test accuracy before and after, and the alignment curve during optimization.

Whatever you choose, load `quantum_kernel_dataset.pkl` for the base-run split indices, the fitted preprocessing objects, and the kernel matrices. The `config` dict inside tells you exactly which run produced it, including the entangling scale the validation sweep chose. Keep the same train/validation/test rows when comparing against the base run.

In [ ]:
# Your extension starts here.
